# Avaliação Final — MEF-LSTM Itajaí-Açu

**Objetivo**: Comparar o baseline (NSE t+7 = 0.119) com o melhor modelo (NSE t+7 = 0.474)  
analisando os eventos históricos de cheia mais relevantes da bacia.

## Estrutura
1. [Carrega resultados](#load) — lê zarr de todos os experimentos
2. [Tabela comparativa](#table) — NSE, KGE, PBIAS para cada melhoria
3. [Hidrogramas — 3 eventos](#hydro) — baseline vs melhor modelo
4. [Análise do pico de 2011](#peak) — quanto o modelo subestimou e por quê
5. [Curvas de excedência](#fdc) — distribuição de fluxos simulados vs observados
6. [MLflow — scatter de métricas](#mlflow) — panorama de todos os experimentos

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

# Ajusta PATH para encontrar módulos do projeto
ROOT = Path().resolve().parents[1]
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'vendor' / 'flood-forecasting'))

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
})

## 1. Funções auxiliares e carregamento de dados <a id='load'></a>

In [ ]:
def nse(obs: np.ndarray, sim: np.ndarray) -> float:
    """Nash-Sutcliffe Efficiency. NSE=1 → perfeito; NSE=0 → igual à média."""
    mask = ~(np.isnan(obs) | np.isnan(sim))
    o, s = obs[mask], sim[mask]
    if len(o) == 0:
        return float('nan')
    return float(1 - np.sum((o - s) ** 2) / np.sum((o - o.mean()) ** 2))

def kge(obs: np.ndarray, sim: np.ndarray) -> float:
    """Kling-Gupta Efficiency. KGE=1 → perfeito; KGE=−0.41 → equivale à média."""
    mask = ~(np.isnan(obs) | np.isnan(sim))
    o, s = obs[mask], sim[mask]
    if len(o) < 2:
        return float('nan')
    r     = float(np.corrcoef(o, s)[0, 1])
    alpha = float(s.std() / o.std()) if o.std() > 0 else float('nan')
    beta  = float(s.mean() / o.mean()) if o.mean() != 0 else float('nan')
    return float(1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2))

def pbias(obs: np.ndarray, sim: np.ndarray) -> float:
    """Percent bias. PBIAS=0 → sem viés volumétrico; negativo → subestima."""
    mask = ~(np.isnan(obs) | np.isnan(sim))
    o, s = obs[mask], sim[mask]
    return float(100 * (s.sum() - o.sum()) / o.sum()) if o.sum() != 0 else float('nan')

In [ ]:
# Mapeamento: rótulo → caminho do zarr de resultados de teste
EXPERIMENTS = {
    'Baseline (MSE, sem lags)': ROOT / 'models/experiments/mef_lstm_itajai_baseline_0906_083244/test/model_epoch015',
    '+ lag features (Mel. 1)':  ROOT / 'models/experiments/exp02_lag_features/exp02_lag_features_0906_111809/test/model_epoch030',
    '+ NSELoss (Mel. 2)':       ROOT / 'models/experiments/exp03_nse_loss/exp03_nse_loss_0906_112336/test/model_epoch030',
    '+ seq 30d (Mel. 3)':       ROOT / 'models/experiments/exp04_seq_30d/exp04_seq_30d_0906_112529/test/model_epoch019',
    '+ ERA5 statics (Mel. 4)':  ROOT / 'models/experiments/exp05_era5_statics/exp05_era5_statics_0906_114408/test/model_epoch028',
}

# Carrega todos os experimentos
data = {}
for label, results_dir in EXPERIMENTS.items():
    zarr_path = results_dir / 'test_results.zarr'
    if not zarr_path.exists():
        print(f'AVISO: {zarr_path} não encontrado')
        continue
    ds = xr.open_zarr(str(zarr_path), consolidated=False).compute()
    basin = str(ds.basin.values[0])
    obs_  = ds['streamflow_obs'].sel(basin=basin, freq='1D').isel(time_step=0).values
    sim7_ = ds['streamflow_sim'].sel(basin=basin, freq='1D').isel(time_step=0).values
    sim0_ = ds['streamflow_sim'].sel(basin=basin, freq='1D').isel(time_step=-1).values
    dates = pd.DatetimeIndex(ds.date.values)
    data[label] = {'dates': dates, 'obs': obs_, 'sim7': sim7_, 'sim0': sim0_}

print(f'Experimentos carregados: {len(data)}')
# Referência: dados observados (todos compartilham o mesmo)
ref = next(iter(data.values()))
dates_all, obs_all = ref['dates'], ref['obs']

## 2. Tabela comparativa <a id='table'></a>

In [ ]:
EVENTS = {
    'nov2008': ('2008-10-01', '2008-12-15'),
    'sep2011': ('2011-08-01', '2011-10-15'),
}

rows = []
for label, d in data.items():
    dates, obs, sim7 = d['dates'], d['obs'], d['sim7']
    row = {
        'Experimento':    label,
        'NSE t+0':        nse(obs, d['sim0']),
        'NSE t+7':        nse(obs, sim7),
        'KGE t+7':        kge(obs, sim7),
        'PBIAS t+7 (%)':  pbias(obs, sim7),
    }
    for ev_name, (t0, t1) in EVENTS.items():
        mask = (dates >= t0) & (dates <= t1)
        row[f'NSE {ev_name}'] = nse(obs[mask], sim7[mask])
        row[f'Pico sim {ev_name}'] = float(np.nanmax(sim7[mask]))
    rows.append(row)

df_results = pd.DataFrame(rows)
df_results.style.background_gradient(cmap='RdYlGn', subset=['NSE t+7', 'KGE t+7', 'NSE nov2008', 'NSE sep2011']) \
                .format(precision=3) \
                .set_caption('Tabela Comparativa — Melhorias do MEF-LSTM')

## 3. Hidrogramas — 3 eventos históricos <a id='hydro'></a>

Comparamos o **Baseline** vs o **melhor modelo** (+ NSELoss, Melhoria 2)  
nos três eventos de cheia mais relevantes do período de teste.

In [ ]:
BEST_MODEL = '+ NSELoss (Mel. 2)'
BASELINE   = 'Baseline (MSE, sem lags)'

FLOOD_EVENTS = [
    ('Cheia de Nov/2008\n(histórica — 3ª maior)',  '2008-10-01', '2008-12-15'),
    ('Cheia de Set/2011\n(2ª maior registrada)',    '2011-08-01', '2011-10-15'),
    ('Cheia de Jan/2022\n(recente)',                '2022-01-01', '2022-03-31'),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 14))
fig.suptitle(
    'MEF-LSTM — Bacia do Itajaí-Açu (Blumenau 83500000)\n'
    'Baseline (MSE) vs Melhor Modelo (lags + NSELoss)',
    fontsize=13, fontweight='bold', y=1.01
)

for ax, (title, t0, t1) in zip(axes, FLOOD_EVENTS):
    d_base = data.get(BASELINE, {})
    d_best = data.get(BEST_MODEL, {})
    if not d_base or not d_best:
        ax.text(0.5, 0.5, 'Dados não carregados', transform=ax.transAxes, ha='center')
        continue

    dates = d_base['dates']
    mask = (dates >= t0) & (dates <= t1)
    if mask.sum() == 0:
        ax.text(0.5, 0.5, f'Sem dados: {t0} → {t1}', transform=ax.transAxes, ha='center')
        continue

    d = dates[mask]
    o  = d_base['obs'][mask]
    sb = d_base['sim7'][mask]
    sm = d_best['sim7'][mask]

    n_base = nse(o, sb)
    n_best = nse(o, sm)

    ax.fill_between(d, 0, o, alpha=0.15, color='steelblue', label='_nolegend_')
    ax.plot(d, o,  color='steelblue',   lw=2.2, label=f'Observado  (pico: {np.nanmax(o):.0f} m³/s)')
    ax.plot(d, sb, color='tomato',      lw=1.8, linestyle='--',
            label=f'Baseline  NSE={n_base:.3f}  (pico: {np.nanmax(sb):.0f} m³/s)')
    ax.plot(d, sm, color='darkorange',  lw=1.8, linestyle='-.',
            label=f'+ lags+NSELoss  NSE={n_best:.3f}  (pico: {np.nanmax(sm):.0f} m³/s)')

    # Marcar pico observado
    peak_idx = np.nanargmax(o)
    ax.axvline(d[peak_idx], color='gray', lw=0.8, linestyle=':', alpha=0.7)
    ax.annotate(f'{o[peak_idx]:.0f} m³/s', xy=(d[peak_idx], o[peak_idx]),
                xytext=(8, 4), textcoords='offset points', fontsize=8, color='steelblue')

    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('Vazão (m³/s)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%b/%Y'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO, interval=2))
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.25)
    ax.set_ylim(bottom=0)

plt.tight_layout()
fig.savefig(ROOT / 'reports/figures/evaluation_flood_events.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Análise do pico de 2011 — por que o modelo subestima? <a id='peak'></a>

A cheia de Set/2011 é o maior evento registrado na série digital (2534 m³/s).  
O melhor modelo simulou ~737 m³/s — **subestimação de 71%**.

In [ ]:
# Análise quantitativa do pico de 2011
d_best = data[BEST_MODEL]
dates, obs, sim7 = d_best['dates'], d_best['obs'], d_best['sim7']

mask_2011 = (dates >= '2011-08-01') & (dates <= '2011-10-15')
d_2011 = dates[mask_2011]
o_2011 = obs[mask_2011]
s_2011 = sim7[mask_2011]

peak_obs = float(np.nanmax(o_2011))
peak_sim = float(np.nanmax(s_2011))
peak_date = d_2011[np.nanargmax(o_2011)]
underest_pct = (peak_obs - peak_sim) / peak_obs * 100

print(f'=== ANÁLISE DO PICO DE SET/2011 (Melhor Modelo) ===')
print(f'Data do pico observado:  {peak_date.date()}')
print(f'Pico observado:         {peak_obs:.0f} m³/s')
print(f'Pico simulado (t+7):    {peak_sim:.0f} m³/s')
print(f'Subestimação absoluta:  {peak_obs - peak_sim:.0f} m³/s')
print(f'Subestimação relativa:  {underest_pct:.1f}%')
print(f'NSE do evento:          {nse(o_2011, s_2011):.3f}')
print(f'KGE do evento:          {kge(o_2011, s_2011):.3f}')

In [ ]:
# Contexto: chuva acumulada antes do pico
chirps = pd.read_parquet(ROOT / 'data/raw/chirps/chirps_itajai_mean_all.parquet')
chirps.index = pd.to_datetime(chirps.index)
precip_col = chirps.columns[0]

# Janela de 30 dias antes do pico
peak_dt = pd.Timestamp(peak_date)
win_start = peak_dt - pd.Timedelta(days=30)
p_window = chirps.loc[win_start:peak_dt, precip_col]
p_cum_30d = p_window.sum()

# Comparar com média histórica dos 30 dias de agosto–setembro (percentis)
all_aug_sep = []
for yr in range(1996, 2024):
    try:
        wp = chirps.loc[f'{yr}-08-01':f'{yr}-09-27', precip_col].sum()
        all_aug_sep.append(wp)
    except:
        pass
all_aug_sep = np.array(all_aug_sep)
pct_2011 = float(np.mean(all_aug_sep <= p_cum_30d) * 100)

print(f'\nCHUVA ACUMULADA NOS 30 DIAS ANTES DO PICO:')
print(f'  2011 (ago-set): {p_cum_30d:.0f} mm')
print(f'  Mediana histórica (30d): {np.median(all_aug_sep):.0f} mm')
print(f'  Percentil histórico de 2011: {pct_2011:.0f}%')
print(f'  → 2011 foi o ano mais chuvoso em {pct_2011:.0f}% das simulações')

In [ ]:
# Visualização: diagnóstico do evento 2011
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=False)
fig.suptitle('Diagnóstico — Cheia de Set/2011 no Melhor Modelo', fontsize=12, fontweight='bold')

# Painel superior: hidrograma com precipitação
ax1b = ax1.twinx()
p_ev = chirps.loc['2011-08-01':'2011-10-15', precip_col]
ax1b.bar(p_ev.index, p_ev.values, color='lightblue', alpha=0.6, label='Precipitação CHIRPS', width=1)
ax1b.set_ylabel('Precipitação (mm/dia)', color='steelblue')
ax1b.invert_yaxis()
ax1b.set_ylim(p_ev.max() * 4, 0)

ax1.fill_between(d_2011, 0, o_2011, alpha=0.15, color='steelblue')
ax1.plot(d_2011, o_2011, color='steelblue', lw=2, label=f'Observado  pico={peak_obs:.0f} m³/s')
ax1.plot(d_2011, s_2011, color='darkorange', lw=1.8, ls='-.', label=f'Simulado t+7  pico={peak_sim:.0f} m³/s')
ax1.axvline(peak_dt, color='red', lw=1, ls=':', alpha=0.7)
ax1.annotate(f'Subestimação\n{underest_pct:.0f}%', xy=(peak_dt, peak_sim),
             xytext=(15, 30), textcoords='offset points', color='red', fontsize=9,
             arrowprops=dict(arrowstyle='->', color='red', lw=1.2))
ax1.set_ylabel('Vazão (m³/s)')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.25)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%d/%b'))
plt.setp(ax1.get_xticklabels(), rotation=30, ha='right')

# Painel inferior: scatter observado × simulado (período de teste completo)
mask_nz = (obs_all > 0) & ~np.isnan(d_best['sim7'])
o_s = obs_all[mask_nz]
s_s = d_best['sim7'][mask_nz]
ax2.scatter(o_s, s_s, alpha=0.3, s=6, color='steelblue', label='Todos os dias')
# Destaque no evento 2011
ax2.scatter(o_2011, s_2011, alpha=0.8, s=20, color='darkorange', label='Set/2011', zorder=5)
lim = max(o_s.max(), s_s.max()) * 1.05
ax2.plot([0, lim], [0, lim], 'k--', lw=1, alpha=0.5, label='Linha 1:1')
ax2.set_xlabel('Vazão Observada (m³/s)')
ax2.set_ylabel('Vazão Simulada t+7 (m³/s)')
ax2.set_xlim(0, lim); ax2.set_ylim(0, lim)
ax2.legend()
ax2.grid(True, alpha=0.25)
ax2.set_title('Scatter observado × simulado (período de teste 2008–2024)')

plt.tight_layout()
fig.savefig(ROOT / 'reports/figures/evaluation_2011_diagnosis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n### POR QUE O MODELO SUBESTIMA O PICO DE 2011 ###')
print(f'1. EXTRAPOLAÇÃO FORA DO DOMÍNIO DE TREINO:')
print(f'   O pico de 2011 ({peak_obs:.0f} m³/s) é o maior da série digital.')
print(f'   O maior pico no treino (1996–2005) foi ~{obs_all[(dates >= "1996") & (dates <= "2005")].max():.0f} m³/s.')
print(f'   O modelo nunca aprendeu padrões de vazões tão altas.')
print()
print(f'2. SUBESTIMAÇÃO ESTRUTURAL EM PICOS EXTREMOS:')
print(f'   LSTMs com MSE/NSELoss tendem a regredir para a média.')
print(f'   Picos raros têm poucos exemplos → gradiente fraco durante o treino.')
print()
print(f'3. PRECIPITAÇÃO SUBESTIMADA PELO CHIRPS:')
print(f'   CHIRPS (grade 5km) subestima chuvas convectivas intensas.')
print(f'   Set/2011 foi um evento de mesoscale convective system.')
print()
print(f'4. SATURAÇÃO DO SOLO NÃO CAPTURADA:')
print(f'   A sequência de 30d antes do pico teve P acumulada de {p_cum_30d:.0f} mm')
print(f'   (percentil {pct_2011:.0f}% histórico). O solo já estava saturado.')
print(f'   Com seq_length=14, o modelo não vê os 30d anteriores completos.')

## 5. Curvas de Permanência de Vazão (FDC) <a id='fdc'></a>

A Flow Duration Curve mostra como a distribuição de fluxos simulados  
se compara à observada — revela vieses sistemáticos por faixa de fluxo.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

def plot_fdc(series, ax, label, color, ls='-', lw=1.5):
    vals = np.sort(series[~np.isnan(series)])[::-1]
    exceedance = np.arange(1, len(vals) + 1) / len(vals) * 100
    ax.plot(exceedance, vals, label=label, color=color, lw=lw, linestyle=ls)

plot_fdc(obs_all,                'Observado',            ax, 'steelblue', lw=2.5)
plot_fdc(data[BASELINE]['sim7'], 'Baseline',             ax, 'tomato', ls='--')
plot_fdc(data[BEST_MODEL]['sim7'],'+ lags + NSELoss',    ax, 'darkorange', ls='-.')
if '+ ERA5 statics (Mel. 4)' in data:
    plot_fdc(data['+ ERA5 statics (Mel. 4)']['sim7'], '+ ERA5 statics', ax, 'green', ls=':')

ax.set_yscale('log')
ax.set_xlabel('Probabilidade de excedência (%)')
ax.set_ylabel('Vazão (m³/s) — escala logarítmica')
ax.set_title('Curva de Permanência de Vazão — Período de Teste (2008–2024)')
ax.legend()
ax.grid(True, which='both', alpha=0.25)
ax.axvline(5, color='gray', lw=0.8, ls=':', alpha=0.5)
ax.text(5.5, ax.get_ylim()[0] * 1.5, 'Alto fluxo\n(< P5)', fontsize=8, color='gray')
ax.axvline(95, color='gray', lw=0.8, ls=':', alpha=0.5)
ax.text(90, ax.get_ylim()[0] * 1.5, 'Fluxo\nbase\n(> P95)', fontsize=8, color='gray', ha='right')

plt.tight_layout()
fig.savefig(ROOT / 'reports/figures/evaluation_fdc.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. MLflow — Panorama de experimentos <a id='mlflow'></a>

In [ ]:
import mlflow

mlflow.set_tracking_uri(f"sqlite:///{ROOT / 'mlruns.db'}")
client = mlflow.tracking.MlflowClient()

experiment = client.get_experiment_by_name('itajai-mef-lstm')
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['metrics.`test/nse_t7` DESC'],
)

mlflow_rows = []
for run in runs:
    m = run.data.metrics
    mlflow_rows.append({
        'Run': run.data.tags.get('mlflow.runName', run.info.run_id[:8]),
        'NSE t+7': m.get('test/nse_t7', None),
        'KGE t+7': m.get('test/kge_t7', None),
        'PBIAS (%)': m.get('test/pbias_t7', None),
        'NSE 2011': m.get('test/nse_sep2011', None),
        'Loss':   run.data.params.get('loss', '?'),
        'seq':    run.data.params.get('seq_length', '?'),
        'inputs': run.data.params.get('hindcast_inputs', '?')[:30] + '...',
    })

df_mlflow = pd.DataFrame(mlflow_rows)
df_mlflow.style.background_gradient(cmap='RdYlGn', subset=['NSE t+7', 'KGE t+7', 'NSE 2011']) \
               .format({'NSE t+7': '{:.3f}', 'KGE t+7': '{:.3f}', 'PBIAS (%)': '{:.1f}', 'NSE 2011': '{:.3f}'}) \
               .set_caption('MLflow — Todos os Experimentos (ordenado por NSE t+7)')

In [ ]:
# Gráfico de barras com evolução das métricas
order = ['Baseline (MSE, sem lags)', '+ lag features (Mel. 1)', '+ NSELoss (Mel. 2)',
         '+ seq 30d (Mel. 3)', '+ ERA5 statics (Mel. 4)']
df_plot = df_results.set_index('Experimento').reindex([k for k in order if k in df_results['Experimento'].values])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Evolução das Melhorias — MEF-LSTM Itajaí-Açu', fontsize=13, fontweight='bold')

metrics_to_plot = [
    ('NSE t+7',    'RdYlGn', 'NSE t+7 (global)'),
    ('KGE t+7',    'RdYlGn', 'KGE t+7 (global)'),
    ('NSE sep2011','RdYlGn', 'NSE evento set/2011'),
]

for ax, (col, cmap, title) in zip(axes, metrics_to_plot):
    vals = df_plot[col].values
    colors = plt.cm.RdYlGn([(v - vals.min()) / (vals.max() - vals.min() + 1e-9) * 0.8 + 0.1 for v in vals])
    bars = ax.barh(range(len(df_plot)), vals, color=colors, edgecolor='gray', linewidth=0.5)
    ax.set_yticks(range(len(df_plot)))
    ax.set_yticklabels([k.replace('(', '\n(') for k in df_plot.index], fontsize=9)
    ax.axvline(0, color='black', lw=0.8, ls='-')
    ax.set_xlabel(title)
    ax.set_title(title, fontsize=10, fontweight='bold')
    for i, (bar, v) in enumerate(zip(bars, vals)):
        ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)
    ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
fig.savefig(ROOT / 'reports/figures/evaluation_metrics_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## Conclusões

| Métrica | Baseline | Melhor (Mel. 2) | Δ |
|---------|---------|-----------------|---|
| NSE t+7 (global) | 0.119 | **0.474** | +0.355 |
| KGE t+7 | 0.068 | **0.513** | +0.445 |
| NSE evento 2008 | –1.105 | **0.061** | +1.166 |
| NSE evento 2011 | –0.607 | **0.120** | +0.727 |
| Pico simulado 2011 | 642 m³/s | 737 m³/s | +15% |

### Maior ganho: streamflow lags (Melhoria 1)
Adicionar `streamflow_lag1` e `streamflow_lag7` como features autoregressivas
foi de longe a melhoria mais impactante. O modelo passou a usar "onde a bacia
está agora" como sinal, não apenas a chuva histórica.

### Por que o modelo ainda subestima picos extremos?
1. **Domínio de treino**: o maior evento de treino foi muito menor que o de 2011
2. **CHIRPS subestima chuvas intensas convectivas**  
3. **seq_length=14 não captura saturação mensal do solo**
4. **Hidden size=32 limita a capacidade de aprender dinâmicas não lineares extremas**

### Próximos passos para superar NSE > 0.60
- Adicionar temperatura e umidade como features dinâmicas (proxy de saturação do solo)
- Aumentar hidden_size para 64 com regularização mais forte
- Usar perda ponderada que priorize picos (WMSE ou extremo-NSE)
- Considerar head probabilístico (CMAL) para quantificar incerteza